# Sanity check — does inference match how I trained?

After notebooks 01–03 I still want a **tiny end-to-end proof** that:

1. `build_stage_feature_payload(...)` builds the exact dictionary keys the classifier expects.
2. If I actually ran **`python model_train.py`** (reads `cleanedGambling.csv`, writes `poker_models.pkl` + `feature_names.pkl`), I can score a toy flop without the Streamlit UI.

If the pickle files aren’t there yet, this cell still prints how many features got assembled — useful when debugging a mismatch.

Whenever **`feature_contracts.STAGE_FEATURES`** changes, re-run **`python model_train.py`** so `feature_names.pkl` matches — old pickles will raise missing-feature errors at inference.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import joblib

from scripts.features.poker_hand_strength import build_stage_feature_payload
from scripts.models.stage_win_predictor import predict_stage_win_probability

# Made-up table spot — AhKd on a monotone-ish flop with some chips in the middle
payload = build_stage_feature_payload(
    "flop",
    hole_cards=["Ah", "kd"],
    board_cards=["Js", "9h", "2c"],
    total_bet=12.0,
    current_pot=40.0,
    position="BTN",
    hero_stack=180.0,
    table_stacks=[200.0, 180.0, 220.0],
    big_blind=2.0,
)
print("Built payload with", len(payload), "numeric keys")

model_path = ROOT / "poker_models.pkl"
feat_path = ROOT / "feature_names.pkl"
if model_path.exists() and feat_path.exists():
    _ = joblib.load(model_path)
    p = predict_stage_win_probability(
        "flop",
        payload,
        model_path=str(model_path),
        feature_path=str(feat_path),
    )
    print("Model says P(win) ~", round(float(p), 4))
else:
    print("No artifacts yet — train first: `python model_train.py` from repo root.")

### Why this matters
The Streamlit poker tab calls the same `predict_stage_win_probability` helper. If this notebook errors with "missing features", training code / UI / saved `feature_names.pkl` drifted — better to catch it here than live.